# Lab 3DGS với Nerfstudio · Splatfacto

Dùng lại `dataset.zip` và `split.json` trong `MyDrive/3dgs_lab`. Chọn runtime GPU T4. Chạy lần lượt để train baseline 7.000 vòng, xuất PLY và xem mô hình. Có thể đặt `STEPS = 500` cho lượt thử nhanh.

In [ ]:
from pathlib import Path
import json, os, shutil, subprocess
from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/3dgs_lab")
WORK = Path("/content/nerfstudio_lab")
WORK.mkdir(exist_ok=True)
DATA = WORK / "dataset"
OUTPUT = DRIVE / "nerfstudio"
OUTPUT.mkdir(exist_ok=True)
STEPS = 7000
DOWNSCALE = 2
ENV = WORK / "env"

def run(*args, env=None):
    command = [str(arg) for arg in args]
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, errors="replace", env=env) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
        if process.wait():
            raise subprocess.CalledProcessError(process.returncode, command)


## 1. Cài môi trường riêng

Phiên bản ghim là Python 3.10, PyTorch 2.1.2/cu118 và Nerfstudio 1.1.5. Toolkit và trình biên dịch nằm trong môi trường riêng. Lần train đầu sẽ biên dịch CUDA của gsplat.

In [ ]:
run("nvidia-smi")
mamba = WORK / "bin/micromamba"
if not mamba.exists():
    archive = WORK / "micromamba.tar.bz2"
    run("curl", "-fLsS", "-o", archive, "https://micro.mamba.pm/api/micromamba/linux-64/latest")
    run("tar", "-xjf", archive, "-C", WORK, "bin/micromamba")

env = os.environ.copy()
env["MAMBA_ROOT_PREFIX"] = str(WORK / "mamba")
env["MAX_JOBS"] = "2"
env["CUDA_HOME"] = str(ENV)
env["TORCH_CUDA_ARCH_LIST"] = "7.5"
# Avoid the Inductor driver-library lookup on Colab. gsplat still uses CUDA.
env["TORCHDYNAMO_DISABLE"] = "1"

def ns(*args):
    run(mamba, "run", "-p", ENV, *args, env=env)

marker = WORK / "setup-complete"
if not marker.exists():
    run(mamba, "create", "-y", "-p", ENV,
        "-c", "nvidia/label/cuda-11.8.0", "-c", "conda-forge",
        "python=3.10", "pip", "cuda-toolkit=11.8", "gxx_linux-64=11", env=env)
    ns("python", "-m", "pip", "install", "torch==2.1.2+cu118", "torchvision==0.16.2+cu118",
       "--index-url", "https://download.pytorch.org/whl/cu118")
    # PyTorch 2.1 imports pkg_resources when gsplat builds its extension.
    ns("python", "-m", "pip", "install", "setuptools==80.9.0",
       "numpy==1.26.4", "nerfstudio==1.1.5")
    ns("python", "-c", "import torch, gsplat; import torch.utils.cpp_extension; assert torch.cuda.is_available(); print(torch.__version__, gsplat.__version__)")
    ns("nvcc", "--version")
    marker.touch()


## 2. Dùng lại dữ liệu COLMAP và tập holdout

Dữ liệu được đọc bằng parser COLMAP của Nerfstudio. Notebook dùng danh sách tên ảnh để giữ đúng split, kể cả khi thứ tự ID ảnh của COLMAP khác thứ tự tên file.

In [ ]:
archive = DRIVE / "dataset.zip"
split_path = DRIVE / "split.json"
if not (DATA / "sparse/0/images.bin").is_file():
    if not archive.is_file():
        raise FileNotFoundError(f"Không thấy {archive}. Hãy lưu dữ liệu từ ô COLMAP của notebook cũ.")
    DATA.mkdir(exist_ok=True)
    shutil.unpack_archive(str(archive), str(DATA))
for name in ("cameras.bin", "images.bin", "points3D.bin"):
    if not (DATA / "sparse/0" / name).is_file():
        raise FileNotFoundError(f"Dataset thiếu sparse/0/{name}.")

split = json.loads(split_path.read_text(encoding="utf-8"))
train, holdout = split["train"], split["holdout"]
names = train + holdout
if not train or not holdout or len(set(names)) != len(names):
    raise ValueError("Danh sách train và holdout phải có ảnh và không được trùng nhau.")
missing = [name for name in names if not (DATA / "images" / name).is_file()]
if missing:
    raise FileNotFoundError(f"Ảnh trong split.json không khớp với dataset.zip: {missing[:5]}")
for part, filenames in (("train", train), ("val", holdout), ("test", holdout)):
    (DATA / f"{part}_list.txt").write_text("\n".join(filenames) + "\n", encoding="utf-8")
print(f"{len(train)} train, {len(holdout)} holdout")


## 3. Huấn luyện bằng Splatfacto

Train baseline mặc định 7.000 vòng. Mỗi lượt có thư mục riêng trên Drive. Giảm kích thước ảnh theo hệ số 2 để thử trên T4. Tắt tối ưu lại pose camera cho lượt so sánh với GraphDECO.

In [ ]:
from PIL import Image

if DOWNSCALE > 1:
    resized_dir = DATA / f"images_{DOWNSCALE}"
    created = 0
    for name in train + holdout:
        target = resized_dir / name
        if target.is_file():
            continue
        target.parent.mkdir(parents=True, exist_ok=True)
        with Image.open(DATA / "images" / name) as image:
            size = (image.width // DOWNSCALE, image.height // DOWNSCALE)
            temporary = target.with_suffix(target.suffix + ".tmp")
            image.resize(size, Image.Resampling.LANCZOS).save(
                temporary, format=image.format, quality=95
            )
            temporary.replace(target)
            created += 1
    print(f"Ảnh giảm kích thước đã sẵn sàng ({created} ảnh mới): {resized_dir}", flush=True)

from datetime import datetime, timezone

stamp = datetime.now(timezone.utc).strftime("%Y-%m-%d_%H%M%S_%f")
experiment = f"baseline-{STEPS}"
ns("ns-train", "splatfacto",
   "--data", DATA,
   "--output-dir", OUTPUT,
   "--experiment-name", experiment,
   "--timestamp", stamp,
   "--max-num-iterations", STEPS,
   "--steps-per-save", min(1000, STEPS),
   "--vis", "tensorboard",
   "--pipeline.model.camera-optimizer.mode", "off",
   "--pipeline.datamanager.cache-images", "cpu",
   "colmap", "--colmap-path", "sparse/0", "--downscale-factor", DOWNSCALE)
CONFIG = OUTPUT / experiment / "splatfacto" / stamp / "config.yml"
assert CONFIG.is_file(), f"Không tìm thấy cấu hình: {CONFIG}"
print("Cấu hình để mở lại mô hình:", CONFIG)


## 4. Đánh giá và xuất PLY

`ns-eval` đo trên tập holdout. `ns-export gaussian-splat` xuất PLY. Nếu mở phiên khác, chạy mục 1–2, rồi đặt `CONFIG` thành đường dẫn đã in ở mục 3 trước khi chạy ô này.

In [ ]:
export = CONFIG.parent / "export"
export.mkdir(exist_ok=True)
ns("ns-eval", "--load-config", CONFIG, "--output-path", export / "metrics.json")
ns("ns-export", "gaussian-splat", "--load-config", CONFIG, "--output-dir", export)
print((export / "metrics.json").read_text(encoding="utf-8"))
print("PLY:", export / "splat.ply")


## 5. Xem mô hình bằng chuột

Tải `splat.ply` từ Drive rồi mở trong [SuperSplat Editor](https://superspl.at/editor). Kéo để xoay, cuộn để zoom. Cách này phù hợp khi Colab không có kết nối cổng cho Viewer.

Nếu chạy trên máy chủ GPU có SSH, dùng `ns-viewer --load-config <config.yml>` trên máy chủ và chuyển tiếp cổng 7007 bằng SSH để dùng Viewer gốc của Nerfstudio.

Các bước tách vật thể, chuẩn hóa pivot và render chung với cảnh đích được thiết kế trong `README.md`. PLY xuất ở mục 4 chứa cả cảnh nhìn thấy trong video.